# Project 2 — Deliverable 2
## Hyperparameter Tuning, Class Imbalance, and Final Model Selection (3-Model Comparison)

This notebook restores the full scope of Deliverable 2 — comparing **Logistic Regression, SVM (RBF), and Random Forest** — after the original attempt was interrupted by SVM's training time on the full ~630K-row dataset.

**Approach:** tuning and the final model fit use a stratified 50,000-row sample of the training set (same technique used in Deliverable 1, section 3), so all three models — including SVM — finish in a reasonable time. Evaluation is done on the full held-out test set.

This notebook is self-contained: it reloads `CrimeDataCleaned.csv` and rebuilds the Top-15 crime-type dataset the same way Deliverable 1 does, so it doesn't depend on variables from another notebook's session.

## 0. Data Loading & Feature Preparation

(Same feature engineering as Deliverable 1, so results are directly comparable.)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from scipy.stats import loguniform, randint
from time import time

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)

In [2]:
# Load cleaned dataset (output of 01_data_cleaning.ipynb)
df = pd.read_csv("CrimeDataCleaned.csv")

# Feature Engineering: Time Columns & Create Top-15 Subset (same as Deliverable 1)
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'])
df['TIME OCC'] = df['TIME OCC'].astype(str).str.zfill(4)
df['Occ_Hour'] = df['TIME OCC'].str[:2].astype(int)
df['Occ_Year'] = df['DATE OCC'].dt.year
df['Occ_Month'] = df['DATE OCC'].dt.month
df['Occ_Day'] = df['DATE OCC'].dt.day
df['Occ_DayOfWeek'] = df['DATE OCC'].dt.dayofweek

df = df.dropna(subset=['Crm Cd'])
top_15 = df['Crm Cd'].value_counts().nlargest(15).index
df_top = df[df['Crm Cd'].isin(top_15)].copy()

df_top.shape

(791009, 40)

In [3]:
# Feature lists (identical to Deliverable 1)
numeric_features = [
    'Vict Age', 'Rpt Dist No', 'Premis Cd', 'Weapon Used Cd', 'LAT', 'LON',
    'Occ_Year', 'Occ_Month', 'Occ_Day', 'Occ_DayOfWeek', 'Occ_Hour',
    'AREA', 'Part 1-2'
]
categorical_features = ['Vict Sex', 'Vict Descent', 'Status']

numeric_features = [c for c in numeric_features if c in df_top.columns]
categorical_features = [c for c in categorical_features if c in df_top.columns]

for col in numeric_features:
    df_top[col] = pd.to_numeric(df_top[col], errors='coerce')

le_y = LabelEncoder()
df_top['CrmCd_label'] = le_y.fit_transform(df_top['Crm Cd'])

X = df_top[numeric_features + categorical_features]
y = df_top['CrmCd_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Full train size:", X_train.shape, " Test size:", X_test.shape)

Full train size: (632807, 16)  Test size: (158202, 16)


In [4]:
# Preprocessing pipeline (identical structure to Deliverable 1)
numeric_tr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_tr = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocess = ColumnTransformer(transformers=[
    ('num', numeric_tr, numeric_features),
    ('cat', categorical_tr, categorical_features)
])

> **Note on sampling:** SVM (RBF) training time grows roughly quadratically with the number of rows. "
On the full training set (~630K rows) it did not finish in reasonable time. Following the same approach "
used in Deliverable 1 (section 3), we tune and fit all three models on a stratified 50,000-row sample of "
the training data, then evaluate on the full test set. This keeps the comparison fair (all three models "
see the same training data) and tractable.

In [5]:
# Stratified subsample for tuning + final model fitting (same technique as Deliverable 1)
X_train_small, _, y_train_small, _ = train_test_split(
    X_train, y_train,
    train_size=20000,
    stratify=y_train,
    random_state=RANDOM_STATE
)

# Fit preprocessing on the sample, transform sample + full test set
preprocess.fit(X_train_small)
X_train_proc = preprocess.transform(X_train_small)
X_test_proc = preprocess.transform(X_test)

print("Sampled train size:", X_train_proc.shape, " Test size:", X_test_proc.shape)

Sampled train size: (20000, 40)  Test size: (158202, 40)


---
## 1. Hyperparameter Explanation

In [6]:
hyperparam_table = pd.DataFrame({
    "Model": [
        "Logistic Regression", "Logistic Regression",
        "SVM (RBF)", "SVM (RBF)",
        "Random Forest", "Random Forest", "Random Forest"
    ],
    "Hyperparameter": [
        "C", "penalty",
        "C", "gamma",
        "n_estimators", "max_depth", "max_features"
    ],
    "Description": [
        "Controls regularization strength (smaller = stronger regularization).",
        "Type of regularization applied (L1 or L2).",
        "Controls margin hardness; larger values increase model complexity.",
        "Controls curvature of decision boundary; larger = more flexible boundary.",
        "Number of decision trees in the forest.",
        "Maximum tree depth; deeper trees risk overfitting.",
        "Number of features considered at each split."
    ]
})
hyperparam_table

,Model,Hyperparameter,Description
0,Logistic Regression,C,Controls regularization strength (smaller = st...
1,Logistic Regression,penalty,Type of regularization applied (L1 or L2).
2,SVM (RBF),C,Controls margin hardness; larger values increa...
3,SVM (RBF),gamma,Controls curvature of decision boundary; large...
4,Random Forest,n_estimators,Number of decision trees in the forest.
5,Random Forest,max_depth,Maximum tree depth; deeper trees risk overfitt...
6,Random Forest,max_features,Number of features considered at each split.


### Presentation Notes (English)
- **Logistic Regression (C):** Smaller C → stronger regularization → simpler model → reduced overfitting.
- **SVM (C, gamma):** Larger C → complex boundaries → higher risk of overfitting. Larger gamma → more curved decision boundary.
- **Random Forest (n_estimators, max_depth):** More trees → more stable model but slower. Deeper trees → possible overfitting.

---
## 2. Hyperparameter Tuning: GridSearchCV vs RandomizedSearchCV

We tune all three models on the 50,000-row sample. **`scoring="f1_weighted"` is used throughout** (not the binary-default `"f1"`), since this is a 15-class classification problem.

In [7]:
results_tuning = []

# === 2.1 Logistic Regression (속도 개선 버전) ===
log_reg = LogisticRegression(solver="saga", max_iter=1000, random_state=RANDOM_STATE)
param_grid_lr = {"C": [0.1, 1, 10], "penalty": ["l1", "l2"]}

start = time()
grid_lr = GridSearchCV(log_reg, param_grid_lr, cv=3, scoring="f1_weighted", n_jobs=-1)
grid_lr.fit(X_train_proc, y_train_small)
grid_time_lr = time() - start

param_dist_lr = {"C": loguniform(1e-3, 1e3), "penalty": ["l1", "l2"]}

start = time()
rand_lr = RandomizedSearchCV(log_reg, param_dist_lr, n_iter=10, cv=3, scoring="f1_weighted",
                              random_state=RANDOM_STATE, n_jobs=-1)
rand_lr.fit(X_train_proc, y_train_small)
rand_time_lr = time() - start

results_tuning.append({"Model": "Logistic Regression", "Search": "Grid",
                        "Best Params": grid_lr.best_params_, "CV Score (f1_weighted)": grid_lr.best_score_, "Time (s)": grid_time_lr})
results_tuning.append({"Model": "Logistic Regression", "Search": "Random",
                        "Best Params": rand_lr.best_params_, "CV Score (f1_weighted)": rand_lr.best_score_, "Time (s)": rand_time_lr})

pd.DataFrame(results_tuning)

,Model,Search,Best Params,CV Score (f1_weighted),Time (s)
0,Logistic Regression,Grid,"{'C': 10, 'penalty': 'l2'}",0.444670,85.793912
1,Logistic Regression,Random,"{'C': 98.77700294007911, 'penalty': 'l2'}",0.444839,86.226310


In [8]:
# === 2.2 SVM (RBF) ===
svc = SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
param_grid_svc = {"C": [1, 10], "gamma": [0.01, 0.1]}

start = time()
grid_svc = GridSearchCV(svc, param_grid_svc, cv=3, scoring="f1_weighted", n_jobs=-1)
grid_svc.fit(X_train_proc, y_train_small)
grid_time_svc = time() - start

param_dist_svc = {"C": loguniform(1e-2, 1e2), "gamma": loguniform(1e-4, 1e0)}

start = time()
rand_svc = RandomizedSearchCV(svc, param_dist_svc, n_iter=8, cv=3, scoring="f1_weighted",
                               random_state=RANDOM_STATE, n_jobs=-1)
rand_svc.fit(X_train_proc, y_train_small)
rand_time_svc = time() - start

results_tuning.append({"Model": "SVM (RBF)", "Search": "Grid",
                        "Best Params": grid_svc.best_params_, "CV Score (f1_weighted)": grid_svc.best_score_, "Time (s)": grid_time_svc})
results_tuning.append({"Model": "SVM (RBF)", "Search": "Random",
                        "Best Params": rand_svc.best_params_, "CV Score (f1_weighted)": rand_svc.best_score_, "Time (s)": rand_time_svc})

pd.DataFrame(results_tuning[-2:])

,Model,Search,Best Params,CV Score (f1_weighted),Time (s)
0,SVM (RBF),Grid,"{'C': 1, 'gamma': 0.1}",0.488585,94.150773
1,SVM (RBF),Random,"{'C': 2.5378155082656657, 'gamma': 0.067965780...",0.498458,241.912425


In [9]:
# === 2.3 Random Forest ===
rf = RandomForestClassifier(random_state=RANDOM_STATE)
param_grid_rf = {"n_estimators": [100, 200], "max_depth": [None, 10], "max_features": ["sqrt"]}

start = time()
grid_rf = GridSearchCV(rf, param_grid_rf, cv=3, scoring="f1_weighted", n_jobs=-1)
grid_rf.fit(X_train_proc, y_train_small)
grid_time_rf = time() - start

param_dist_rf = {
    "n_estimators": randint(100, 300),
    "max_depth": [None, 5, 10],
    "max_features": ["sqrt", "log2"]
}

start = time()
rand_rf = RandomizedSearchCV(rf, param_dist_rf, n_iter=10, cv=3, scoring="f1_weighted",
                              random_state=RANDOM_STATE, n_jobs=-1)
rand_rf.fit(X_train_proc, y_train_small)
rand_time_rf = time() - start

results_tuning.append({"Model": "Random Forest", "Search": "Grid",
                        "Best Params": grid_rf.best_params_, "CV Score (f1_weighted)": grid_rf.best_score_, "Time (s)": grid_time_rf})
results_tuning.append({"Model": "Random Forest", "Search": "Random",
                        "Best Params": rand_rf.best_params_, "CV Score (f1_weighted)": rand_rf.best_score_, "Time (s)": rand_time_rf})

tuning_summary = pd.DataFrame(results_tuning)
tuning_summary

,Model,Search,Best Params,CV Score (f1_weighted),Time (s)
0,Logistic Regression,Grid,"{'C': 10, 'penalty': 'l2'}",0.444670,85.793912
1,Logistic Regression,Random,"{'C': 98.77700294007911, 'penalty': 'l2'}",0.444839,86.226310
2,SVM (RBF),Grid,"{'C': 1, 'gamma': 0.1}",0.488585,94.150773
3,SVM (RBF),Random,"{'C': 2.5378155082656657, 'gamma': 0.067965780...",0.498458,241.912425
4,Random Forest,Grid,"{'max_depth': None, 'max_features': 'sqrt', 'n...",0.542448,8.198836
5,Random Forest,Random,"{'max_depth': None, 'max_features': 'sqrt', 'n...",0.542471,14.439857


### Presentation Notes
- **Grid Search** → Exhaustive search → more accurate but slower.
- **Random Search** → Faster, explores fewer combinations → efficient for large search spaces.
- Compare the table above across all three models: which search method found a better (or equally good) score faster, for each model?

---
## 3. Data Balancing (Imbalanced Classification)

We compare the class distribution before and after SMOTE and Random Undersampling, applied to the sampled training set used for tuning and fitting.

In [10]:
orig_counts = pd.Series(y_train_small).value_counts()
pd.DataFrame({"count": orig_counts, "ratio": orig_counts / orig_counts.sum()})

,count,ratio
CrmCd_label,,
10,2913,0.14565
11,1892,0.09460
3,1606,0.08030
6,1581,0.07905
13,1545,0.07725
2,1463,0.07315
8,1358,0.06790
1,1353,0.06765
12,1181,0.05905


In [11]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train_proc, y_train_small)

# 크기가 너무 커지면 다시 3만 행으로 줄이기 (클래스 비율은 유지됨)
if len(X_train_sm) > 30000:
    X_train_sm, _, y_train_sm, _ = train_test_split(
        X_train_sm, y_train_sm,
        train_size=30000,
        stratify=y_train_sm,
        random_state=RANDOM_STATE
    )

sm_counts = pd.Series(y_train_sm).value_counts()
pd.DataFrame({"count": sm_counts, "ratio": sm_counts/sm_counts.sum()})

,count,ratio
CrmCd_label,,
14,2000,0.066667
4,2000,0.066667
3,2000,0.066667
1,2000,0.066667
11,2000,0.066667
7,2000,0.066667
6,2000,0.066667
8,2000,0.066667
2,2000,0.066667


In [12]:
rus = RandomUnderSampler(random_state=RANDOM_STATE)
X_train_ru, y_train_ru = rus.fit_resample(X_train_proc, y_train_small)

ru_counts = pd.Series(y_train_ru).value_counts()
pd.DataFrame({"count": ru_counts, "ratio": ru_counts / ru_counts.sum()})

,count,ratio
CrmCd_label,,
0,642,0.066667
1,642,0.066667
2,642,0.066667
3,642,0.066667
4,642,0.066667
5,642,0.066667
6,642,0.066667
7,642,0.066667
8,642,0.066667


---
## 4. Final Model Evaluation & Selection

Each of the three tuned models (using its best `GridSearchCV` parameters) is evaluated on **Original / SMOTE / Undersampling** training data — 9 combinations total — always tested on the same held-out test set. Metrics used: Accuracy, Macro F1, Weighted F1, and weighted one-vs-rest ROC-AUC (appropriate for multi-class, imbalanced data — see Deliverable 1's justification for F1 over accuracy).

In [13]:
def evaluate_model(estimator, X_tr, y_tr, X_ev, y_ev, model_name, setting_name):
    clf = clone(estimator)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_ev)
    y_proba = clf.predict_proba(X_ev) if hasattr(clf, "predict_proba") else None

    row = {
        "Model": model_name,
        "Resampling": setting_name,
        "Accuracy": accuracy_score(y_ev, y_pred),
        "F1 (macro)": f1_score(y_ev, y_pred, average="macro", zero_division=0),
        "F1 (weighted)": f1_score(y_ev, y_pred, average="weighted", zero_division=0),
    }
    if y_proba is not None:
        row["ROC-AUC (OVR, weighted)"] = roc_auc_score(y_ev, y_proba, multi_class="ovr", average="weighted")
    else:
        row["ROC-AUC (OVR, weighted)"] = np.nan
    return row

final_lr = grid_lr.best_estimator_
final_svc = grid_svc.best_estimator_
final_rf = grid_rf.best_estimator_

models_final = {
    "Logistic Regression": final_lr,
    "SVM (RBF)": final_svc,
    "Random Forest": final_rf
}

datasets = {
    "Original": (X_train_proc, y_train_small),
    "SMOTE": (X_train_sm, y_train_sm),
    "Undersampling": (X_train_ru, y_train_ru)
}

summary_rows = []
for model_name, estimator in models_final.items():
    for setting_name, (X_tr, y_tr) in datasets.items():
        summary_rows.append(
            evaluate_model(estimator, X_tr, y_tr, X_test_proc, y_test, model_name, setting_name)
        )

summary_df = pd.DataFrame(summary_rows)
summary_df.sort_values(["F1 (weighted)"], ascending=False)

,Model,Resampling,Accuracy,F1 (macro),F1 (weighted),"ROC-AUC (OVR, weighted)"
6,Random Forest,Original,0.587211,0.472960,0.544038,0.942333
7,Random Forest,SMOTE,0.568830,0.483373,0.543498,0.939965
8,Random Forest,Undersampling,0.532617,0.479087,0.524772,0.938866
4,SVM (RBF),SMOTE,0.517212,0.445870,0.499810,0.929109
3,SVM (RBF),Original,0.548546,0.420844,0.496153,0.932997
5,SVM (RBF),Undersampling,0.504855,0.429337,0.481038,0.928850
2,Logistic Regression,Undersampling,0.473433,0.392626,0.447968,0.915036
1,Logistic Regression,SMOTE,0.475797,0.390446,0.446692,0.915432
0,Logistic Regression,Original,0.497579,0.369851,0.445203,0.916326


In [14]:
# Best combination by weighted F1
best_row = summary_df.loc[summary_df["F1 (weighted)"].idxmax()]
print("Best combination (by weighted F1):")
print(best_row)

Best combination (by weighted F1):
Model                      Random Forest
Resampling                      Original
Accuracy                        0.587211
F1 (macro)                       0.47296
F1 (weighted)                   0.544038
ROC-AUC (OVR, weighted)         0.942333
Name: 6, dtype: object


### Final Presentation Notes
- Pick the best model + resampling combination based primarily on **Weighted F1** (overall performance under class imbalance) and **Macro F1** (fairness across all 15 crime types), with ROC-AUC as a secondary check.
- Fill in the actual winning combination and a short interpretation here once this notebook has been run end-to-end.
- Compare against Deliverable 2's original Logistic-Regression-only result (Weighted F1 ≈ 0.456 for LR–SMOTE) to see whether SVM or Random Forest improves on it.